# 🔴 Solution: DMD2 Distribution Matching Loss

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
# ✅ SOLUTION

def dmd_loss(x_gen, pred_real, pred_fake, eps=1e-8):
    # Every dim except the batch dim
    dims = tuple(range(1, x_gen.dim()))

    # Per-sample scale of the teacher's correction — makes the update dimension-free
    normalizer = (pred_real - x_gen).abs().mean(dim=dims, keepdim=True)

    # Score difference = gradient of the reverse KL w.r.t. the generated sample
    grad = (pred_fake - pred_real) / (normalizer + eps)
    grad = torch.nan_to_num(grad)

    # Surrogate loss whose derivative IS that gradient (target carries no graph)
    target = (x_gen - grad).detach()
    return 0.5 * F.mse_loss(x_gen, target)

In [ ]:
# Verify
x = torch.randn(4, 3, 8, 8, requires_grad=True)
pred_real = torch.randn(4, 3, 8, 8)
pred_fake = torch.randn(4, 3, 8, 8)

loss = dmd_loss(x, pred_real, pred_fake)
loss.backward()
normalizer = (pred_real - x.detach()).abs().mean(dim=(1, 2, 3), keepdim=True)
expected = (pred_fake - pred_real) / (normalizer + 1e-8) / x.numel()
print("loss           :", loss.item())
print("matches theory :", torch.allclose(x.grad, expected, atol=1e-6))
print("zero when equal:", dmd_loss(x, pred_real, pred_real).item())

In [ ]:
from torch_judge import check
check("dmd2")